# BrainOmni Feature Extraction

In [18]:
import os
import sys
import json
import numpy as np
import torch

sys.path.insert(0, '/Users/faenegoro/Documents/Neuroimaging/BrainOmni')
sys.path.insert(0, '/Users/faenegoro/Documents/Neuroimaging/meg_ad_project')

from brainomni.model import BrainOmni
from src.preprocessing import preprocess_subject, to_tensor, FS_ORIG, FS_TARGET, SEGMENT_SECS

print('imports ok')
print('MPS available:', torch.backends.mps.is_available())

In [19]:
# Build subject labels: RSID0001-0070 = control (0), RSID0071-0148 = AD (1)
DATA_DIR = '/Users/faenegoro/Documents/Neuroimaging/meg_ad_project/data/meg_mri'

files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('_meg_rest_60sec.mat')])

# 0 = control, 1 = AD
labels = {}
for f in files:
    subject_id = f.split('_')[0]
    number = int(subject_id[4:])
    labels[subject_id] = 0 if number <= 70 else 1

print(f'Total subjects: {len(labels)}')
print(f'Controls: {sum(v == 0 for v in labels.values())}')
print(f'AD: {sum(v == 1 for v in labels.values())}')

Total subjects: 148
Controls: 70
AD: 78


In [20]:
# Load the pretrained BrainOmni model
CKPT_PATH = '/Users/faenegoro/Documents/Neuroimaging/BrainOmni/ckpt_collection/base'

with open(os.path.join(CKPT_PATH, 'model_cfg.json')) as f:
    model_config = json.load(f)

model = BrainOmni(**model_config)
checkpoint = torch.load(os.path.join(CKPT_PATH, 'BrainOmni.pt'), map_location='cpu', weights_only=True)
model.load_state_dict(checkpoint, strict=False)
model.eval()

# Use MPS (Apple Silicon GPU) if available, otherwise CPU
device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
model = model.to(device)

print(f'Model loaded on: {device}')

/opt/miniconda3/envs/brainomni/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Model loaded on: mps


In [21]:
# Test preprocessing on one subject
DATA_DIR = '/Users/faenegoro/Documents/Neuroimaging/meg_ad_project/data/meg_mri'

test_path = os.path.join(DATA_DIR, 'RSID0001_meg_rest_60sec.mat')
signal, pos, sensor_type = preprocess_subject(test_path)

print('Signal shape after resample:', signal.shape)   # expect (271, 15360)
print('Pos shape:', pos.shape)                        # expect (271, 6)
print('Sensor type shape:', sensor_type.shape)        # expect (271,)
print('Signal mean (should be ~0):', signal.mean())
print('Signal std (should be ~1):', signal.std())

In [22]:
test_path = os.path.join(DATA_DIR, 'RSID0001_meg_rest_60sec.mat')
signal, pos, sensor_type = preprocess_subject(test_path)

segment_len = SEGMENT_SECS * FS_TARGET
segment_1 = signal[:, :segment_len]
segment_2 = signal[:, segment_len:segment_len*2]

with torch.no_grad():
    x1, pos1, st1 = to_tensor(segment_1, pos, sensor_type, device)
    features_1 = model.encode(x1, pos1, st1)

    x2, pos2, st2 = to_tensor(segment_2, pos, sensor_type, device)
    features_2 = model.encode(x2, pos2, st2)

print('Features segment 1 shape:', features_1.shape)
print('Features segment 2 shape:', features_2.shape)

feat_1 = features_1.mean(dim=2)
feat_2 = features_2.mean(dim=2)
subject_features = ((feat_1 + feat_2) / 2).squeeze(0)
print('Subject feature shape:', subject_features.shape)

In [23]:
from tqdm import tqdm

DATA_DIR = '/Users/faenegoro/Documents/Neuroimaging/meg_ad_project/data/meg_mri'
files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('_meg_rest_60sec.mat')])

all_features = []
all_labels   = []
all_ids      = []
failed       = []

for filename in tqdm(files):
    subject_id = filename.split('_')[0]
    mat_path   = os.path.join(DATA_DIR, filename)

    try:
        signal, pos, sensor_type = preprocess_subject(mat_path)

        segment_len = SEGMENT_SECS * FS_TARGET
        segment_1 = signal[:, :segment_len]
        segment_2 = signal[:, segment_len:segment_len*2]

        with torch.no_grad():
            x1, pos1, st1 = to_tensor(segment_1, pos, sensor_type, device)
            x2, pos2, st2 = to_tensor(segment_2, pos, sensor_type, device)
            feat_1 = model.encode(x1, pos1, st1).mean(dim=2)
            feat_2 = model.encode(x2, pos2, st2).mean(dim=2)

        subject_feat = ((feat_1 + feat_2) / 2).squeeze(0)
        subject_feat = subject_feat.flatten().cpu().numpy()

        all_features.append(subject_feat)
        all_labels.append(labels[subject_id])
        all_ids.append(subject_id)

    except Exception as e:
        print(f'Failed: {subject_id} — {e}')
        failed.append(subject_id)

X = np.stack(all_features)
y = np.array(all_labels)

print(f'Features shape: {X.shape}')
print(f'Failed subjects: {failed}')

save_dir = '/Users/faenegoro/Documents/Neuroimaging/meg_ad_project/data'
model_name = 'base' if X.shape[1] == 8192 else 'tiny'
np.save(os.path.join(save_dir, f'brainomni_features_{model_name}.npy'), X)
np.save(os.path.join(save_dir, 'labels.npy'), y)
np.save(os.path.join(save_dir, 'subject_ids.npy'), np.array(all_ids))

print(f'Saved as brainomni_features_{model_name}.npy')

In [24]:
import numpy as np
import pandas as pd

X   = np.load('/Users/faenegoro/Documents/Neuroimaging/meg_ad_project/data/brainomni_features.npy')
y   = np.load('/Users/faenegoro/Documents/Neuroimaging/meg_ad_project/data/labels.npy')
ids = np.load('/Users/faenegoro/Documents/Neuroimaging/meg_ad_project/data/subject_ids.npy')

save_dir = '/Users/faenegoro/Documents/Neuroimaging/meg_ad_project/data'

pd.DataFrame({'subject_id': ids}).to_csv(f'{save_dir}/subject_ids.csv', index=False)
pd.DataFrame({'subject_id': ids, 'label': y}).to_csv(f'{save_dir}/labels.csv', index=False)

feature_cols = [f'feature_{i}' for i in range(X.shape[1])]
df_features = pd.DataFrame(X, columns=feature_cols)
df_features.insert(0, 'subject_id', ids)
df_features.to_csv(f'{save_dir}/brainomni_features.csv', index=False)


In [25]:
import pandas as pd

file = pd.read_excel('/Users/faenegoro/Documents/Neuroimaging/meg_ad_project/data/eLife91044_processeddata_scalarmetrics.xlsx')
baseline_features = file[['Z AEC deltatheta', 'Z AEC alpha', 'Z AEC beta', 'Z Pow deltatheta', 'Z Pow alpha', 'Z Pow beta']].values
np.save('/Users/faenegoro/Documents/Neuroimaging/meg_ad_project/data/baseline_features', baseline_features)
print(baseline_features.shape)
print(baseline_features)

(148, 6)
[[ 3.26264797e+00 -3.07131920e+00 -3.06000292e+00  1.26006771e+00
   2.91351922e-01  1.63431255e-01]
 [-5.20596787e-01 -9.91258443e-01  3.03545284e-01  1.03824016e-01
   3.31713844e-01  4.84342319e-01]
 [ 1.43208629e+00 -1.30943961e+00 -1.98932911e+00  1.16585981e+00
   8.45680973e-01  7.91091079e-01]
 [ 6.61049195e-01  8.13985818e-01  1.20592559e-01 -8.52017189e-01
   1.34385657e-01  3.75453842e-01]
 [-2.60185534e-01  1.41579684e+00  3.00487565e-01  6.39857937e-02
  -9.49998075e-01  6.45494109e-01]
 [ 5.95945724e-01 -1.27484329e-01 -5.53998651e-01 -7.64637417e-01
  -1.69565001e+00 -7.55491229e-01]
 [ 5.74715552e-01  7.95748142e-01  1.16486106e+00  8.94709155e-01
  -6.54645876e-01  1.29614030e+00]
 [-3.00140606e-02 -2.39427730e+00 -2.54156767e+00  1.30883681e+00
  -1.49674057e+00  8.07867832e-01]
 [ 2.09547914e-01  9.75563280e-02  1.01147385e+00 -1.87193711e-01
   1.00582872e-01 -1.12420363e+00]
 [-2.61613522e-01  6.31499790e-01  8.16173301e-01 -1.66073689e-01
  -2.25757460e+0